# Ride with GPS Data Retrieval Tool

This notebook retrieves route and ride data from Ride with GPS and stores it in Delta tables.
**Prerequisites:**
Install required libraries in cluster
Set up secrets in Databricks Secret Scope for login credentials
Have a valid Ride with GPS account (ridewithgps.com)

**Prerequisites:**
- Install required libraries in cluster
- Set up secrets in Databricks Secret Scope for login credentials
- Have a valid Ride with GPS account (ridewithgps.com)

## Deployment Notes

### For Production with API Key Authentication:

1. **Create API Client at**: https://ridewithgps.com/settings/developers
2. **Set up secrets**: 
   ```
   databricks secrets create-scope ridewithgps
   databricks secrets put ridewithgps api_key
   databricks secrets put ridewithgps email
   databricks secrets put ridewithgps password
   ```
3. **Get the API key from the created client** and add your email/password to the secrets.

### API Client Setup:
- Application name: "Databricks Data Extraction Tool"  
- Application category: "Other"
- Description: "API client for extracting route and trip data for analysis"
- The API key will be generated after saving the client

### Recommended Job Schedule:
- Routes: Weekly (since routes change less frequently)
- Trips: Daily (for new completed rides)

### Rate Limiting:
- Ride with GPS has API rate limits
- Monitor API responses for rate limit headers
- Implement retry logic with backoff if needed


In [0]:
import requests
import json
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import *
from pyspark.sql.functions import *
import logging
from typing import Dict, List, Optional, Any

# Initialize Spark session (automatically available in Databricks)
spark = SparkSession.builder.getOrCreate()


In [0]:
class RideWithGPSAPI:
    """
    Ride with GPS API client
    
    This client now uses the correct authentication process:
    1. A POST request to /api/v1/auth_tokens.json with email and password to get a user-specific auth_token.
    This corrects the previous attempt that was using the wrong endpoint and method.
    """
    
    def __init__(self, secret_scope: str = "ridewithgps"):
        self.session = requests.Session()
        self.base_url = "https://ridewithgps.com"
        self.api_base_url = "https://ridewithgps.com/api/v1"
        self.secret_scope = secret_scope
        self.is_authenticated = False
        self.user_id = None
        self.auth_token = None
        self.api_key = None
        
        # Set up logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
        # Set up session headers
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Sec-Fetch-Dest': 'empty',
            'Sec-Fetch-Mode': 'cors',
            'Sec-Fetch-Site': 'same-origin',
        })
        
    def authenticate(self) -> bool:
        """
        Authenticate using API key and user email/password.
        
        This method retrieves the API key and user credentials from Databricks
        secrets, and then performs a two-step authentication to get a user-specific
        auth_token.
        """
        try:
            # Get API credentials and user credentials from Databricks secrets
            self.api_key = dbutils.secrets.get(scope=self.secret_scope, key="api_key")
            email = dbutils.secrets.get(scope=self.secret_scope, key="email")
            password = dbutils.secrets.get(scope=self.secret_scope, key="password")
            
            self.logger.info("Attempting to authenticate with Ride with GPS API...")
            
            # Step 1: Set the API key for the session.
            # This needs to be done for all requests.
            self.session.params.update({'api_key': self.api_key})
            
            # Step 2: Authenticate the user with email/password to get the auth_token.
            # The correct endpoint for this is /api/v1/auth_tokens.json
            auth_url = f"{self.api_base_url}/auth_tokens.json"
            
            # The payload for authentication, now with the correct nested 'user' key
            payload = {
                'user': {
                    'email': email,
                    'password': password
                }
            }
            
            # Send a POST request to the correct endpoint
            response = self.session.post(auth_url, json=payload)
            response.raise_for_status() # Raise an exception for bad status codes
            
            auth_data = response.json()
            
            if 'auth_token' in auth_data and 'user' in auth_data['auth_token']:
                self.auth_token = auth_data['auth_token']['auth_token']
                self.user_id = auth_data['auth_token']['user']['id']
                self.is_authenticated = True
                
                # Update the session with the auth_token for subsequent requests
                self.session.params.update({'auth_token': self.auth_token})
                
                self.logger.info(f"✅ Successfully authenticated and retrieved auth token. User ID: {self.user_id}")
                return True
            else:
                self.logger.error("❌ Authentication failed. Could not retrieve user ID or auth token.")
                return False
                
        except requests.exceptions.HTTPError as e:
            self.logger.error(f"❌ HTTP Error during authentication: {e.response.status_code} - {e.response.text}")
            return False
        except Exception as e:
            self.logger.error(f"❌ Authentication error: {str(e)}")
            return False
    
    def get_routes(self, user_id: int = None, page: int = 1, limit: int = 50) -> List[Dict]:
        """
        Retrieve routes from Ride with GPS
        
        Args:
            user_id: User ID to get routes for (defaults to authenticated user)
            page: Page number for pagination
            limit: Number of routes per page
            
        Returns:
            List of route dictionaries and metadata
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        target_user = user_id or self.user_id
        
        if not target_user:
            self.logger.error("User ID is required to get routes for a specific user.")
            return [], {}
            
        url = f"{self.api_base_url}/routes.json"
        
        params = {
            'page': page,
            'limit': limit,
            'user_id': target_user # Pass user_id as a query parameter
        }
        
        try:
            self.logger.info(f"Trying URL: {url} with user_id={target_user}")
            response = self.session.get(url, params=params)
            response.raise_for_status()
            
            data = response.json()
            # The API returns routes in the 'routes' key for this endpoint
            routes = data.get('routes', []) 
            self.logger.info(f"✅ Successfully retrieved {len(routes)} routes from {url}")
            return routes, data.get('meta', {})
                
        except requests.exceptions.RequestException as e:
            self.logger.error(f"❌ Request error for {url}: {str(e)}")
            return [], {}
    
    def get_route_details(self, route_id: int) -> Dict:
        """
        Get detailed information for a specific route including GPX data
        
        Args:
            route_id: ID of the route to retrieve
            
        Returns:
            Dict containing route details
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        url = f"{self.api_base_url}/routes/{route_id}.json"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            return response.json().get('route', {})
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error retrieving route details for {route_id}: {str(e)}")
            return {}
    
    def get_trips(self, user_id: int = None, page: int = 1, limit: int = 50) -> List[Dict]:
        """
        Retrieve completed trips/rides from Ride with GPS
        
        Args:
            user_id: User ID to get trips for (defaults to authenticated user)  
            page: Page number for pagination
            limit: Number of trips per page
            
        Returns:
            List of trip dictionaries and metadata
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        target_user = user_id or self.user_id
        
        if not target_user:
            self.logger.error("User ID is required to get trips for a specific user.")
            return [], {}
        
        url = f"{self.api_base_url}/trips.json"
        
        params = {
            'page': page,
            'limit': limit,
            'user_id': target_user # Pass user_id as a query parameter
        }
        
        try:
            self.logger.info(f"Trying URL: {url} with user_id={target_user}")
            response = self.session.get(url, params=params)
            response.raise_for_status()
                
            data = response.json()
            # The API returns trips in the 'trips' key for this endpoint
            trips = data.get('trips', []) 
            self.logger.info(f"✅ Successfully retrieved {len(trips)} trips from {url}")
            return trips, data.get('meta', {})
                
        except requests.exceptions.RequestException as e:
            self.logger.error(f"❌ Request error for {url}: {str(e)}")
            return [], {}
    
    def get_trip_details(self, trip_id: int) -> Dict:
        """
        Get detailed information for a specific trip
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        url = f"{self.api_base_url}/trips/{trip_id}.json"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            return response.json().get('trip', {})
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error retrieving trip details for {trip_id}: {str(e)}")
            return {}

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Processing Functions

# COMMAND ----------

def get_routes_schema() -> StructType:
    """
    Returns the explicit schema for the routes table.
    """
    return StructType([
        StructField('route_id', LongType(), True),
        StructField('name', StringType(), True),
        StructField('description', StringType(), True),
        StructField('distance', DoubleType(), True),
        StructField('elevation_gain', DoubleType(), True),
        StructField('elevation_loss', DoubleType(), True),
        StructField('difficulty', StringType(), True),
        StructField('surface_type', StringType(), True),
        StructField('created_at', TimestampType(), True),
        StructField('updated_at', TimestampType(), True),
        StructField('is_private', BooleanType(), True),
        StructField('user_id', LongType(), True),
        StructField('locality', StringType(), True),
        StructField('administrative_area', StringType(), True),
        StructField('country_code', StringType(), True),
        StructField('bounding_box_sw_lat', DoubleType(), True),
        StructField('bounding_box_sw_lng', DoubleType(), True),
        StructField('bounding_box_ne_lat', DoubleType(), True),
        StructField('bounding_box_ne_lng', DoubleType(), True),
        StructField('retrieved_at', TimestampType(), True)
    ])

def get_trips_schema() -> StructType:
    """
    Returns the explicit schema for the trips table.
    """
    return StructType([
        StructField('trip_id', LongType(), True),
        StructField('name', StringType(), True),
        StructField('description', StringType(), True),
        StructField('distance', DoubleType(), True),
        StructField('duration', DoubleType(), True),
        StructField('elevation_gain', DoubleType(), True),
        StructField('elevation_loss', DoubleType(), True),
        StructField('avg_speed', DoubleType(), True),
        StructField('max_speed', DoubleType(), True),
        StructField('started_at', TimestampType(), True),
        StructField('is_private', BooleanType(), True),
        StructField('user_id', LongType(), True),
        StructField('route_id', LongType(), True),
        StructField('locality', StringType(), True),
        StructField('administrative_area', StringType(), True),
        StructField('country_code', StringType(), True),
        StructField('retrieved_at', TimestampType(), True)
    ])


def process_routes_to_dataframe(routes_data: List[Dict]) -> pd.DataFrame:
    """
    Convert routes data to a pandas DataFrame with proper data types
    """
    if not routes_data:
        return pd.DataFrame()
    
    # Extract relevant fields from routes
    processed_routes = []
    for route in routes_data:
        processed_route = {
            'route_id': route.get('id'),
            'name': route.get('name'),
            'description': route.get('description'),
            'distance': route.get('distance'),  # in meters
            'elevation_gain': route.get('elevation_gain'),  # in meters
            'elevation_loss': route.get('elevation_loss'),  # in meters
            'difficulty': route.get('difficulty'),
            'surface_type': route.get('surface_type'),
            'created_at': pd.to_datetime(route.get('created_at')),
            'updated_at': pd.to_datetime(route.get('updated_at')),
            'is_private': route.get('is_private', False),
            'user_id': route.get('user_id'),
            'locality': route.get('locality'),
            'administrative_area': route.get('administrative_area'),
            'country_code': route.get('country_code'),
            'bounding_box_sw_lat': route.get('bounding_box', [None, None, None, None])[0],
            'bounding_box_sw_lng': route.get('bounding_box', [None, None, None, None])[1],
            'bounding_box_ne_lat': route.get('bounding_box', [None, None, None, None])[2],
            'bounding_box_ne_lng': route.get('bounding_box', [None, None, None, None])[3],
            'retrieved_at': datetime.now()
        }
        processed_routes.append(processed_route)
    
    return pd.DataFrame(processed_routes)

def process_trips_to_dataframe(trips_data: List[Dict]) -> pd.DataFrame:
    """
    Convert trips data to a pandas DataFrame
    """
    if not trips_data:
        return pd.DataFrame()
    
    processed_trips = []
    for trip in trips_data:
        processed_trip = {
            'trip_id': trip.get('id'),
            'name': trip.get('name'),
            'description': trip.get('description'),
            'distance': trip.get('distance'),  # in meters
            'duration': trip.get('duration'),  # in seconds
            'elevation_gain': trip.get('elevation_gain'),
            'elevation_loss': trip.get('elevation_loss'),
            'avg_speed': trip.get('avg_speed'),
            'max_speed': trip.get('max_speed'),
            'started_at': pd.to_datetime(trip.get('started_at')),
            'is_private': trip.get('is_private', False),
            'user_id': trip.get('user_id'),
            'route_id': trip.get('route_id'),  # If trip was based on a route
            'locality': trip.get('locality'),
            'administrative_area': trip.get('administrative_area'),
            'country_code': trip.get('country_code'),
            'retrieved_at': datetime.now()
        }
        processed_trips.append(processed_trip)
    
    return pd.DataFrame(processed_trips)

def save_to_delta_table(df: pd.DataFrame, table_name: str, schema: StructType, mode: str = "overwrite"):
    """
    Save DataFrame to Delta table in Databricks with an explicit schema.
    """
    print(f"📊 Attempting to save DataFrame to {table_name} with mode='{mode}'")
    
    if df.empty:
        print(f"❌ No data to save to {table_name} - DataFrame is empty")
        return
    
    try:
        # Convert pandas DataFrame to Spark DataFrame using the explicit schema
        print(f"🔄 Converting pandas DataFrame to Spark DataFrame with explicit schema...")
        spark_df = spark.createDataFrame(df, schema=schema)
        
        print(f"🔄 Writing to Delta table {table_name}...")
        # Write to Delta table
        spark_df.write \
            .format("delta") \
            .mode(mode) \
            .option("mergeSchema", "true") \
            .saveAsTable(table_name)
        
        print(f"✅ Successfully saved {len(df)} records to {table_name}")
        
    except Exception as e:
        print(f"❌ Error saving to {table_name}: {str(e)}")
        print(f"📊 DataFrame sample:")
        print(df.head())

# COMMAND ----------

# MAGIC %md
# MAGIC ## Main Extraction Functions

# COMMAND ----------

def extract_all_routes(user_id: int = None, max_pages: int = 10) -> bool:
    """
    Extract all routes from Ride with GPS and save to Delta table
    
    Args:
        user_id: Specific user ID to get routes for (optional)
        max_pages: Maximum number of pages to retrieve
    """
    try:
        # Initialize API client
        rwgps_api = RideWithGPSAPI()
        
        # Authenticate
        if not rwgps_api.authenticate():
            raise Exception("Failed to authenticate with Ride with GPS")
        
        all_routes = []
        page = 1
        
        while page <= max_pages:
            print(f"📄 Retrieving routes page {page}...")
            # Pass user_id to the get_routes function
            routes, meta = rwgps_api.get_routes(user_id=rwgps_api.user_id, page=page, limit=50)
            
            if not routes:
                print(f"No more routes found on page {page}")
                break
            
            all_routes.extend(routes)
            print(f"📊 Found {len(routes)} routes on page {page} (total: {len(all_routes)})")
            
            # Check if there's a next page
            pagination = meta.get('pagination', {})
            if not pagination.get('next_page_url'):
                print("📄 Reached last page")
                break
                
            page += 1
        
        if not all_routes:
            print("❌ No routes found")
            return True
        
        print(f"📊 Processing {len(all_routes)} routes...")
        
        # Process to DataFrame
        df = process_routes_to_dataframe(all_routes)
        
        # Save to Delta table using the new overwrite mode and explicit schema
        save_to_delta_table(df, "ridewithgps.routes", schema=get_routes_schema(), mode="overwrite")
        
        return True
        
    except Exception as e:
        print(f"❌ Error in extract_all_routes: {str(e)}")
        return False

def extract_route_details(route_ids: List[int] = None, sample_size: int = 10) -> bool:
    """
    Extract detailed route information including GPX data
    
    Args:
        route_ids: Specific route IDs to get details for
        sample_size: If no route_ids provided, get details for this many recent routes
    """
    try:
        rwgps_api = RideWithGPSAPI()
        
        if not rwgps_api.authenticate():
            raise Exception("Failed to authenticate with Ride with GPS")
        
        if not route_ids:
            # Get recent routes to sample from
            print(f"📊 Getting sample of {sample_size} routes for detailed extraction...")
            # Pass user_id to the get_routes function
            routes, _ = rwgps_api.get_routes(user_id=rwgps_api.user_id, page=1, limit=sample_size)
            route_ids = [route.get('id') for route in routes if route.get('id')]
        
        route_details = []
        
        for route_id in route_ids:
            print(f"📍 Getting details for route {route_id}...")
            details = rwgps_api.get_route_details(route_id)
            
            if details:
                details['retrieved_at'] = datetime.now()
                route_details.append(details)
            
        if route_details:
            # For detailed routes, we might want to store the full JSON
            df = pd.DataFrame(route_details)
            # The route_details table can be appended to, as it's for specific route details,
            # but we still use the overwrite mode for the full pipeline.
            # Here we'll use append to save the detailed data, as it's a separate process
            save_to_delta_table(df, "ridewithgps.route_details", schema=get_route_details_schema(), mode="append")
        
        return True
        
    except Exception as e:
        print(f"❌ Error in extract_route_details: {str(e)}")
        return False

def get_route_details_schema() -> StructType:
    """
    Returns the explicit schema for the route_details table.
    This schema is simplified to handle the main details and a string-ified JSON for complex data.
    """
    return StructType([
        StructField('id', LongType(), True),
        StructField('name', StringType(), True),
        StructField('created_at', TimestampType(), True),
        StructField('updated_at', TimestampType(), True),
        StructField('description', StringType(), True),
        StructField('distance', DoubleType(), True),
        StructField('elevation_gain', DoubleType(), True),
        StructField('elevation_loss', DoubleType(), True),
        StructField('surface_type', StringType(), True),
        StructField('difficulty', StringType(), True),
        StructField('user_id', LongType(), True),
        StructField('gpx', StringType(), True), # Store GPX as a string to avoid complex types
        StructField('retrieved_at', TimestampType(), True)
    ])


def extract_trips(user_id: int = None, max_pages: int = 5) -> bool:
    """
    Extract trip/ride data from Ride with GPS
    """
    try:
        rwgps_api = RideWithGPSAPI()
        
        if not rwgps_api.authenticate():
            raise Exception("Failed to authenticate with Ride with GPS")
        
        all_trips = []
        page = 1
        
        while page <= max_pages:
            print(f"📄 Retrieving trips page {page}...")
            # Pass user_id to the get_trips function
            trips, meta = rwgps_api.get_trips(user_id=rwgps_api.user_id, page=page, limit=50)
            
            if not trips:
                print(f"No more trips found on page {page}")
                break
            
            all_trips.extend(trips)
            print(f"📊 Found {len(trips)} trips on page {page} (total: {len(all_trips)})")
            
            # Check if there's a next page
            pagination = meta.get('pagination', {})
            if not pagination.get('next_page_url'):
                print("📄 Reached last page")
                break
                
            page += 1
        
        if all_trips:
            df = process_trips_to_dataframe(all_trips)
            # Save to Delta table using the new overwrite mode and explicit schema
            save_to_delta_table(df, "ridewithgps.trips", schema=get_trips_schema(), mode="overwrite")
        
        return True
        
    except Exception as e:
        print(f"❌ Error in extract_trips: {str(e)}")
        return False

# COMMAND ----------

# MAGIC %md
# MAGIC ## Test Authentication

# COMMAND ----------

def test_authentication():
    """
    Test Ride with GPS authentication and explore available endpoints
    """
    try:
        rwgps_api = RideWithGPSAPI()
        
        if rwgps_api.authenticate():
            print("✅ Authentication successful!")
            print(f"User ID: {rwgps_api.user_id}")
            
            # Test different endpoints to see what's available
            print("\n🔍 Testing different endpoints...")
            
            # We now know the correct endpoints. Let's test them.
            test_endpoints = [
                "/users/current.json", # This still works
                f"/routes.json?user_id={rwgps_api.user_id}", # New, corrected endpoint
                f"/trips.json?user_id={rwgps_api.user_id}",  # New, corrected endpoint
            ]
            
            for endpoint in test_endpoints:
                if endpoint is None:
                    continue
                    
                try:
                    url = f"{rwgps_api.api_base_url}{endpoint}"
                    response = rwgps_api.session.get(url)
                    
                    if response.status_code == 200:
                        data = response.json()
                        # This check is now more robust to see if the expected keys are in the response
                        if 'routes' in data or 'trips' in data or 'user' in data:
                            print(f"✅ {endpoint}: Status {response.status_code} - Keys: {list(data.keys())}")
                        else:
                            print(f"❌ {endpoint}: Status {response.status_code} - Unexpected keys: {list(data.keys())}")
                    else:
                        print(f"❌ {endpoint}: Status {response.status_code}")
                        if response.status_code == 401:
                            print(f"    → Unauthorized (may need different permissions)")
                        elif response.status_code == 404:
                            print(f"    → Not found (endpoint may not exist)")
                            
                except Exception as e:
                    print(f"❌ {endpoint}: Error - {str(e)}")
            
            print("\nYou can now run the full data extraction.")
            return True
        else:
            print("❌ Authentication failed!")
            print("Please check your API key, email, and password in the secret scope.")
            return False
            
    except Exception as e:
        print(f"❌ Authentication error: {str(e)}")
        return False

# Uncomment to test authentication
# test_authentication()



## Main

In [0]:

# Create the database first
spark.sql("CREATE DATABASE IF NOT EXISTS ridewithgps")
print("✅ Created ridewithgps database")

# Run the data extraction
if __name__ == "__main__":
    print("🚴 Starting Ride with GPS data extraction...")
    
    # Test authentication first
    if not test_authentication():
        print("❌ Authentication failed, stopping extraction")
    else:
        # Extract routes (your main interest)
        print("\n📍 Extracting routes...")
        success_routes = extract_all_routes(max_pages=5)
        
        # Extract a sample of route details
        print("\n📋 Extracting route details...")  
        success_details = extract_route_details(sample_size=5)
        
        # Extract trips (optional)
        print("\n🚴 Extracting trips...")
        success_trips = extract_trips(max_pages=3)
        
        if success_routes and success_details and success_trips:
            print("✅ Ride with GPS data extraction completed successfully!")
        else:
            print("❌ Some errors occurred during extraction")



## Data Quality Checks

In [0]:
def run_data_quality_checks():
    """
    Run basic data quality checks on the extracted data
    """
    try:
        print("=== ROUTES TABLE ===")
        routes_df = spark.table("ridewithgps.routes")
        routes_count = routes_df.count()
        print(f"Total routes in table: {routes_count}")
        
        if routes_count > 0:
            routes_df.select("name", "distance", "elevation_gain", "created_at").show(5)
            
            # Summary statistics
            print("\n📊 Route Statistics:")
            routes_df.select(
                mean("distance").alias("avg_distance_m"),
                max("distance").alias("max_distance_m"),
                mean("elevation_gain").alias("avg_elevation_m"),
                count("*").alias("total_routes")
            ).show()
        
        print("\n=== TRIPS TABLE ===")
        trips_df = spark.table("ridewithgps.trips")
        trips_count = trips_df.count()
        print(f"Total trips in table: {trips_count}")
        
        if trips_count > 0:
            trips_df.select("name", "distance", "duration", "started_at").show(5)
        
        print("\n=== ROUTE DETAILS TABLE ===")
        try:
            details_df = spark.table("ridewithgps.route_details")
            details_count = details_df.count()
            print(f"Total detailed routes: {details_count}")
        except:
            print("Route details table not found")
        
    except Exception as e:
        print(f"❌ Error running data quality checks: {str(e)}")

# Uncomment to run data quality checks after extraction
run_data_quality_checks()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Deployment Notes
# MAGIC 
# MAGIC ### For Production with API Key Authentication:
# MAGIC 
# MAGIC 1. **Create API Client at**: https://ridewithgps.com/settings/developers
# MAGIC 2. **Set up secrets**: 
# MAGIC    ```
# MAGIC    databricks secrets create-scope ridewithgps
# MAGIC    databricks secrets put ridewithgps api_key
# MAGIC    databricks secrets put ridewithgps email
# MAGIC    databricks secrets put ridewithgps password
# MAGIC    ```
# MAGIC 3. **Get the API key from the created client** and add your email/password to the secrets.
# MAGIC 
# MAGIC ### API Client Setup:
# MAGIC - Application name: "Databricks Data Extraction Tool"  
# MAGIC - Application category: "Other"
# MAGIC - Description: "API client for extracting route and trip data for analysis"
# MAGIC - The API key will be generated after saving the client
# MAGIC 
# MAGIC ### Recommended Job Schedule:
# MAGIC - Routes: Weekly (since routes change less frequently)
# MAGIC - Trips: Daily (for new completed rides)
# MAGIC 
# MAGIC ### Rate Limiting:
# MAGIC - Ride with GPS has API rate limits
# MAGIC - Monitor API responses for rate limit headers
# MAGIC - Implement retry logic with backoff if needed


## Example Usage


# Create the database first
```
spark.sql("CREATE DATABASE IF NOT EXISTS ridewithgps")
print("✅ Created ridewithgps database")
```
# Run the data extraction
```
if __name__ == "__main__":
    print("🚴 Starting Ride with GPS data extraction...")
    
    # Test authentication first
    if not test_authentication():
        print("❌ Authentication failed, stopping extraction")
    else:
        # Extract routes (your main interest)
        print("\n📍 Extracting routes...")
        success_routes = extract_all_routes(max_pages=5)
        
        # Extract a sample of route details
        print("\n📋 Extracting route details...")  
        success_details = extract_route_details(sample_size=5)
        
        # Extract trips (optional)
        print("\n🚴 Extracting trips...")
        success_trips = extract_trips(max_pages=3)
        
        if success_routes and success_details and success_trips:
            print("✅ Ride with GPS data extraction completed successfully!")
        else:
            print("❌ Some errors occurred during extraction")
```